In [1]:
import os
import cv2
import joblib
import numpy as np

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Main project directory
BASE_PATH = r"C:\Users\hp\Downloads\dataset\dataset"

# Cropped dataset
DATASET_PATH = os.path.join(BASE_PATH, "faces_cropped")

# Final trained model directory
MODEL_PATH = os.path.join(BASE_PATH, "saved_model_cropped_8x8")

# HOG configuration
HOG_ORIENTATIONS = 9
PIXELS_PER_CELL = (8, 8)
CELLS_PER_BLOCK = (2, 2)
BLOCK_NORM = "L2-Hys"

# Image dimensions
IMAGE_SIZE = (100, 100)

print("Project paths and model configuration set!")
print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Model directory exists:", os.path.exists(MODEL_PATH))

Project paths and model configuration set!
Dataset exists: True
Model directory exists: True


In [3]:
model = joblib.load(
    os.path.join(MODEL_PATH, "face_recognition_svm.pkl")
)

label_encoder = joblib.load(
    os.path.join(MODEL_PATH, "label_encoder.pkl")
)

print("Model loaded successfully!")
print("Available celebrities:")
print(label_encoder.classes_)

Model loaded successfully!
Available celebrities:
['Aamir' 'Ajay' 'Akshay' 'Alia' 'Amitabh' 'Deepika' 'Disha' 'Farhan'
 'Ileana']


In [4]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    print("Error loading face detection model!")
else:
    print("Face detection initialized successfully!")

Face detection initialized successfully!


In [5]:
def predict_celebrity(image_path):

    image = cv2.imread(image_path)

    if image is None:
        return {
            "status": "error",
            "message": "Unable to read image."
        }

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    if len(faces) == 0:
        return {
            "status": "error",
            "message": "No face detected. Please upload another image."
        }

    # Select the largest detected region
    x, y, w, h = max(
        faces,
        key=lambda face: face[2] * face[3]
    )

    face = gray[y:y+h, x:x+w]

    face = cv2.resize(face, IMAGE_SIZE)

    face_normalized = face / 255.0

    features = hog(
        face_normalized,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=PIXELS_PER_CELL,
        cells_per_block=CELLS_PER_BLOCK,
        block_norm=BLOCK_NORM
    )

    prediction = model.predict(features.reshape(1, -1))[0]

    celebrity_name = label_encoder.inverse_transform(
        [prediction]
    )[0]

    return {
        "status": "success",
        "celebrity": str(celebrity_name),
        "detected_faces": len(faces)
    }


print("Prediction function created successfully!")

Prediction function created successfully!


In [6]:
test_image = r"C:\Users\Hp\Desktop\illeana.jpeg"

result = predict_celebrity(test_image)

print(result)

{'status': 'success', 'celebrity': 'Amitabh', 'detected_faces': 2}


In [7]:
test_image = os.path.join(
    DATASET_PATH,
    "Ileana",
    os.listdir(os.path.join(DATASET_PATH, "Ileana"))[0]
)

result = predict_celebrity(test_image)

print(result)

{'status': 'success', 'celebrity': 'Ileana', 'detected_faces': 1}


In [8]:
%pip install python-multipart
